# 00 · Setup & data audit
Run once at the start of the project. It
1. downloads GWFSS_v1.0 (labelled split) and converts masks to class-index PNGs,
2. **freezes the data splits** (`splits/splits.json`) — never regenerate them after experiments start,
3. audits the annotations (empty masks, institutes, stem share),
4. benchmarks training speed per backbone on your GPU to plan the Colab budget.

Class ids (verified against `GlobalWheat/GWFSS_model_v1.0/class.json`): 0 background, 1 head, **2 stem**, 3 leaf.

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
import shutil, json
os.makedirs(f'{PROJECT}/splits', exist_ok=True)
shutil.copy(f'{DATA_ROOT}/manifest.csv', f'{PROJECT}/splits/manifest.csv')
if os.path.exists(SPLIT_FILE):
    print('Splits already frozen - NOT regenerating:', SPLIT_FILE)
else:
    splits = data.make_splits(manifest, seed=42)
    data.save_splits(splits, SPLIT_FILE)
    print('Splits written:', SPLIT_FILE)
splits = json.load(open(SPLIT_FILE))
for k in ['random', 'region'] + [k for k in splits if k.startswith('loio_')]:
    print(k, {s: len(v) for s, v in splits[k].items()})

## Annotation audit

In [ ]:
import pandas as pd
m = manifest.copy()
m['stem_share'] = m.stem_px / (m.height * m.width)
audit = m.groupby('institute').agg(images=('name', 'count'), empty_masks=('empty_mask', 'sum'),
                                   images_with_stem=('stem_px', lambda s: (s > 0).sum()),
                                   mean_stem_share=('stem_share', 'mean'), unknown_px=('unknown_px', 'sum'))
display(audit)
print('Images with an all-background mask:'); print(m[m.empty_mask].name.tolist())
audit.to_csv(f'{PROJECT}/splits/annotation_audit.csv')

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
PALETTE = np.array([[0,0,0],[50,255,132],[50,132,255],[214,255,50]], np.uint8)
def show(name):
    img = cv2.cvtColor(cv2.imread(f'{DATA_ROOT}/images/{name}'), cv2.COLOR_BGR2RGB)
    msk = cv2.imread(f'{DATA_ROOT}/masks/{name}', cv2.IMREAD_UNCHANGED)
    col = PALETTE[np.where(msk == 255, 0, msk)]
    ov = img.copy(); ov[msk == 2] = [255, 0, 255]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, im, t in zip(ax, [img, col, ov], [name, 'mask', 'stem overlay (magenta)']):
        a.imshow(im); a.set_title(t, fontsize=8); a.axis('off')
    plt.show()
for inst, grp in m[m.stem_px > 0].groupby('institute'):
    show(grp.sort_values('stem_px').iloc[-1]['name'])

## Speed benchmark (plan the Colab budget)
Measures seconds/iteration for each backbone on the current GPU (batch 8, crop 512, AMP). Multiply by 8,000 iterations for the time per run.

In [ ]:
import time, torch
from gwfss_stem.models import BACKBONES, build_model
from gwfss_stem.losses import SegLoss
x = torch.randn(8, 3, 512, 512, device='cuda'); y = torch.randint(0, 4, (8, 512, 512), device='cuda')
for name in BACKBONES:
    model = build_model(name).cuda(); opt = torch.optim.AdamW(model.parameters(), 1e-4)
    crit = SegLoss('cldice'); scaler = torch.amp.GradScaler('cuda')
    for i in range(25):
        if i == 5: torch.cuda.synchronize(); t = time.time()
        with torch.autocast('cuda'):
            out = model(x)
        loss, _ = crit(out.float(), y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    torch.cuda.synchronize(); spi = (time.time() - t) / 20
    print(f'{name:22s} {spi:.3f} s/it  -> ~{spi*8000/60:.0f} min per run  (peak mem {torch.cuda.max_memory_allocated()/1e9:.1f} GB)')
    del model, opt; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()